In [ ]:
%run ./utils_common

In [ ]:
logger = setup_logger("TotalAllPurposeSpendsReporter")

In [ ]:
dbutils.widgets.text("catalog", "", "CATALOG")
dbutils.widgets.text("schema", "", "SCHEMA")
dbutils.widgets.text("overlap_days", "3", "Overlap days (min 2)")

In [ ]:
# =======================================================
# Total All-Purpose Spends Client
# =======================================================
# Sibling of TotalJobSpendsClient. Differences:
#   * source DBU table: dbspend360_all_purpose_dbu_cost
#                       (keyed (cluster_id, user_id, usage_date))
#   * target table:     dbspend360_total_all_purpose_spends
#   * join condition:   includes `currency` (guards multi-currency fan-out
#                       on the cloud_cost_explorer side; fix relative to
#                       the job-cluster pipeline) and uses LEFT join so
#                       DBU rows whose cloud counterpart has not landed
#                       still flow through with cloud_cost = 0.0.
#   * no apportionment: under owner attribution (plan §3.3) every
#                       (cluster_id, usage_date) maps 1:1 to one user_id,
#                       so cluster cloud_cost flows directly. The reconciliation
#                       invariant is asserted post-merge.
#   * MERGE key:        (cluster_id, user_id, usage_date)
#   * data_security_mode is denormalized through from the DBU table so the
#     UI can render attribution-quality badges without an extra join.
class TotalAllPurposeSpendsClient:

    TABLE_NAME = "dbspend360_total_all_purpose_spends"

    def __init__(
        self,
        audit_table: str,
        cloud_cost_table: str,
        databricks_cost_table: str,
        target_table: str,
        error_log_table: str,
        overlap_days: int,
        logger=None,
    ):
        self.audit_table = audit_table
        self.cloud_cost_table = cloud_cost_table
        self.databricks_cost_table = databricks_cost_table
        self.target_table = target_table
        self.error_log_table = error_log_table
        self.overlap_days = overlap_days
        self.logger = logger or logging.getLogger("TotalAllPurposeSpendsClient")

    def _log_errors(self, dbu_df, cloud_df):
        # Mirror of databricks_job_spends_app's _log_errors. user_id (and
        # data_security_mode) is captured in the raw_record JSON column on
        # the DBU-only half, so no error-log DDL change is needed (plan §5.4).
        # `job_id` / `run_id` slots are filled with NULL since this pipeline
        # operates on all-purpose compute where those columns do not apply.
        dbu_only = (
            dbu_df.alias("d")
            .join(
                cloud_df.alias("a"),
                on=(
                    (F.col("d.cluster_id") == F.col("a.cluster_id")) &
                    (F.col("d.usage_date") == F.col("a.cost_incurred_date")) &
                    (F.col("d.currency")   == F.col("a.currency"))
                ),
                how="left_anti"
            )
        )

        if not dbu_only.isEmpty():
            dbu_err = (
                dbu_only
                .select(
                    F.lit("DBR_DBU").alias("source_system"),
                    F.lit("NO_MATCH_cloud_COST").alias("error_type"),
                    F.col("d.cluster_id").alias("cluster_id"),
                    F.lit(None).cast("string").alias("job_id"),
                    F.lit(None).cast("string").alias("run_id"),
                    F.col("d.usage_date").alias("usage_date"),
                    F.col("d.currency").alias("currency"),
                    F.lit("No matching cloud VM cost row for this DBU usage").alias("error_detail"),
                    F.to_json(F.struct("d.*")).alias("raw_record"),
                )
                .withColumn("created_at", F.lit(datetime.now(timezone.utc)))
            )
            _safe_append(dbu_err, self.error_log_table)

        cloud_only = (
            cloud_df.alias("a")
            .join(
                dbu_df.alias("d"),
                on=(
                    (F.col("a.cluster_id") == F.col("d.cluster_id")) &
                    (F.col("a.cost_incurred_date") == F.col("d.usage_date")) &
                    (F.col("a.currency") == F.col("d.currency"))
                ),
                how="left_anti"
            )
        )

        if not cloud_only.isEmpty():
            cloud_err = (
                cloud_only
                .select(
                    F.lit("cloud_COST").alias("source_system"),
                    F.lit("NO_MATCH_DBR_DBU").alias("error_type"),
                    F.col("a.cluster_id").alias("cluster_id"),
                    F.lit(None).cast("string").alias("job_id"),
                    F.lit(None).cast("string").alias("run_id"),
                    F.lit(None).cast("date").alias("usage_date"),
                    F.col("a.currency").alias("currency"),
                    F.lit("No matching DBR DBU cost row for this cloud VM cost").alias("error_detail"),
                    F.to_json(F.struct("a.*")).alias("raw_record"),
                )
                .withColumn("created_at", F.lit(datetime.now(timezone.utc)))
            )
            _safe_append(cloud_err, self.error_log_table)

    def _assert_reconciliation(self, start_dt, end_dt):
        # Plan §3.3 invariant: for every (cluster_id, usage_date) row written
        # to the target, SUM(cloud_cost) by (cluster_id, usage_date, currency)
        # must equal dbspend360_cloud_cost_explorer.cloud_cost for the same
        # (cluster_id, cost_incurred_date, currency) within 0.01 USD.
        #
        # Under v1 owner attribution there is exactly one user_id per
        # (cluster_id, usage_date), so SUM == single-row value. Summing keeps
        # this assertion forward-compatible with v2 DBU-proportional
        # apportionment without changing the formulation.
        mismatch_df = spark.sql(f"""
            WITH spend AS (
                SELECT cluster_id, usage_date, currency,
                       SUM(cloud_cost) AS spend_cloud_cost
                FROM {self.target_table}
                WHERE usage_date BETWEEN '{start_dt}' AND '{end_dt}'
                GROUP BY cluster_id, usage_date, currency
            ),
            explorer AS (
                SELECT cluster_id,
                       cost_incurred_date AS usage_date,
                       currency,
                       cloud_cost AS explorer_cloud_cost
                FROM {self.cloud_cost_table}
                WHERE cost_incurred_date BETWEEN '{start_dt}' AND '{end_dt}'
            )
            SELECT s.cluster_id, s.usage_date, s.currency,
                   s.spend_cloud_cost, e.explorer_cloud_cost,
                   ABS(s.spend_cloud_cost - e.explorer_cloud_cost) AS diff
            FROM spend s
            JOIN explorer e
              ON s.cluster_id = e.cluster_id
             AND s.usage_date = e.usage_date
             AND s.currency   = e.currency
            WHERE ABS(s.spend_cloud_cost - e.explorer_cloud_cost) > 0.01
        """)

        mismatch_count = mismatch_df.count()
        if mismatch_count == 0:
            self.logger.info(
                f"Reconciliation OK: cloud_cost in {self.target_table} matches "
                f"{self.cloud_cost_table} ± 0.01 USD for {start_dt} → {end_dt}"
            )
            return

        # Persist mismatches to error_log so a failed run leaves a paper trail
        # even when we re-raise below.
        err_df = (
            mismatch_df
            .select(
                F.lit("RECONCILIATION").alias("source_system"),
                F.lit("CLOUD_COST_MISMATCH").alias("error_type"),
                F.col("cluster_id"),
                F.lit(None).cast("string").alias("job_id"),
                F.lit(None).cast("string").alias("run_id"),
                F.col("usage_date"),
                F.col("currency"),
                F.concat(
                    F.lit("cloud_cost mismatch: spend="),
                    F.col("spend_cloud_cost").cast("string"),
                    F.lit(", explorer="),
                    F.col("explorer_cloud_cost").cast("string"),
                    F.lit(", diff="),
                    F.col("diff").cast("string"),
                ).alias("error_detail"),
                F.to_json(F.struct(
                    "cluster_id", "usage_date", "currency",
                    "spend_cloud_cost", "explorer_cloud_cost", "diff",
                )).alias("raw_record"),
            )
            .withColumn("created_at", F.lit(datetime.now(timezone.utc)))
        )
        try:
            _safe_append(err_df, self.error_log_table)
        except Exception as e:
            self.logger.exception(
                f"Failed to write reconciliation mismatches to error log: {e}"
            )

        sample = mismatch_df.limit(5).collect()
        sample_str = "; ".join(
            f"(cluster={r['cluster_id']}, date={r['usage_date']}, "
            f"spend={r['spend_cloud_cost']}, explorer={r['explorer_cloud_cost']}, "
            f"diff={r['diff']:.4f})"
            for r in sample
        )
        raise DataQualityError(
            f"Reconciliation invariant violated: {mismatch_count} "
            f"(cluster_id, usage_date) pairs differ from {self.cloud_cost_table} "
            f"by > 0.01 USD. Sample: {sample_str}"
        )

    def build_total_all_purpose_spends(self):
        start_dt = end_dt = datetime.now(timezone.utc).date()
        try:
            start_dt, end_dt = get_date_window(self.audit_table, self.TABLE_NAME, self.overlap_days)

            valid, msg = validate_date_window(start_dt, end_dt)
            if not valid:
                raise DataQualityError(msg)

            self.logger.info(
                f"Building dbspend360_total_all_purpose_spends for {start_dt} → {end_dt}"
            )

            ensure_cost_columns(self.target_table, logger=self.logger)

            cloud_df = (
                spark.table(self.cloud_cost_table)
                    .alias("cc")
                    .filter(
                        (F.col("cost_incurred_date") >= F.lit(start_dt)) &
                        (F.col("cost_incurred_date") <= F.lit(end_dt))
                    )
            )
            dbu_df = (
                spark.table(self.databricks_cost_table)
                    .alias("dbu")
                    .filter(
                        (F.col("usage_date") >= F.lit(start_dt)) &
                        (F.col("usage_date") <= F.lit(end_dt))
                    )
            )

            if dbu_df.limit(1).count() == 0:
                self.logger.info(
                    "No all-purpose DBU rows in this date window; nothing to join."
                )
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt,
                    "SUCCESS", 0, "No DBU data in window",
                )
                return

            cc_columns = {c.name for c in spark.table(self.cloud_cost_table).schema}
            has_segmented = "compute_cost" in cc_columns
            has_other = "other_cost" in cc_columns

            # LEFT join with currency in the predicate (plan §5.4). Currency
            # guards against multi-currency fan-out on the cloud side; the
            # job-cluster pipeline has the same latent risk and we fix it
            # here. Rows without a cloud counterpart still flow through with
            # cloud_cost defaulted to 0.0; the unmatched halves are written
            # to the error log via _log_errors.
            joined = dbu_df.join(
                cloud_df,
                on=(
                    (dbu_df["cluster_id"] == cloud_df["cluster_id"]) &
                    (dbu_df["usage_date"] == cloud_df["cost_incurred_date"]) &
                    (dbu_df["currency"]   == cloud_df["currency"])
                ),
                how="left"
            )

            joined = joined.withColumn(
                "final_currency",
                F.coalesce(F.col("dbu.currency"), F.col("cc.currency"))
            )

            joined = joined.withColumn(
                "cloud_cost_filled",
                F.coalesce(F.col("cc.cloud_cost"), F.lit(0.0))
            )

            select_cols = [
                F.col("dbu.cluster_id").alias("cluster_id"),
                F.col("dbu.user_id").alias("user_id"),
                F.col("dbu.usage_date").alias("usage_date"),
                F.col("cloud_cost_filled").alias("cloud_cost"),
                F.col("dbu.databricks_cost").alias("databricks_cost"),
                F.col("final_currency").alias("currency"),
                F.col("dbu.data_security_mode").alias("data_security_mode"),
            ]

            if has_segmented:
                select_cols.extend([
                    F.col("cc.compute_cost").alias("compute_cost"),
                    F.col("cc.storage_cost").alias("storage_cost"),
                    F.col("cc.network_cost").alias("network_cost"),
                ])
            if has_other:
                select_cols.append(F.col("cc.other_cost").alias("other_cost"))

            final_df = joined.select(*select_cols)

            if not has_segmented:
                final_df = (
                    final_df
                    .withColumn("compute_cost", F.lit(None).cast("double"))
                    .withColumn("storage_cost", F.lit(None).cast("double"))
                    .withColumn("network_cost", F.lit(None).cast("double"))
                )
            if not has_other:
                final_df = final_df.withColumn("other_cost", F.lit(None).cast("double"))

            final_df = (
                final_df
                .withColumn(
                    "total_cost",
                    F.coalesce(F.col("cloud_cost"), F.lit(0.0))
                    + F.coalesce(F.col("databricks_cost"), F.lit(0.0)),
                )
                .withColumn("created_at", F.current_timestamp())
                .withColumn("updated_at", F.current_timestamp())
            )
            final_df = safe_cache(final_df)

            row_count = final_df.count()

            validate_source_schema(
                final_df,
                {"cluster_id": "string", "user_id": "string",
                 "usage_date": "date", "cloud_cost": "double",
                 "databricks_cost": "double"},
                self.target_table, self.logger,
            )
            validate_no_negative_costs(
                final_df,
                ["cloud_cost", "databricks_cost", "total_cost",
                 "compute_cost", "storage_cost", "network_cost", "other_cost"],
                self.target_table, self.logger,
            )
            validate_currency_consistency(final_df, "currency", self.target_table, self.logger)

            target = DeltaTable.forName(spark, self.target_table)
            (target.alias("t")
                .merge(
                    final_df.alias("s"),
                    "t.cluster_id = s.cluster_id AND t.user_id = s.user_id "
                    "AND t.usage_date = s.usage_date",
                )
                .whenMatchedUpdate(set={
                    "cloud_cost": "s.cloud_cost",
                    "compute_cost": "s.compute_cost",
                    "storage_cost": "s.storage_cost",
                    "network_cost": "s.network_cost",
                    "other_cost": "s.other_cost",
                    "databricks_cost": "s.databricks_cost",
                    "total_cost": "s.total_cost",
                    "data_security_mode": "s.data_security_mode",
                    "updated_at": "current_timestamp()",
                })
                .whenNotMatchedInsert(values={
                    "cluster_id": "s.cluster_id",
                    "user_id": "s.user_id",
                    "usage_date": "s.usage_date",
                    "cloud_cost": "s.cloud_cost",
                    "compute_cost": "s.compute_cost",
                    "storage_cost": "s.storage_cost",
                    "network_cost": "s.network_cost",
                    "other_cost": "s.other_cost",
                    "databricks_cost": "s.databricks_cost",
                    "currency": "s.currency",
                    "total_cost": "s.total_cost",
                    "data_security_mode": "s.data_security_mode",
                    "created_at": "current_timestamp()",
                    "updated_at": "current_timestamp()",
                })
                .execute()
            )

            safe_unpersist(final_df)
            get_merge_metrics(self.target_table, self.logger)

            validate_post_merge(
                self.target_table, "usage_date",
                start_dt, end_dt, row_count, self.logger,
            )

            # Hard gate: a violation here means the rollup disagrees with the
            # cost-explorer source of truth, which would silently break the
            # §9 acceptance criterion. Raise so the audit row records FAILED;
            # the next run will re-MERGE within the overlap window.
            self._assert_reconciliation(start_dt, end_dt)

            # Error logging should not impact pipeline success.
            try:
                self._log_errors(dbu_df, cloud_df)
            except Exception as e:
                self.logger.exception(
                    f"Error logging failed, but pipeline succeeded: {e}"
                )

            log_audit_run(self.audit_table, self.TABLE_NAME, start_dt, end_dt, "SUCCESS", row_count, "")
            self.logger.info(
                f"Merged {row_count} rows into {self.target_table} "
                f"for {start_dt} → {end_dt}."
            )

        except Exception as e:
            msg = str(e)[:1000]
            self.logger.error(f"Run failed: {msg}")
            try:
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt, "FAILED", 0, msg,
                )
            except Exception:
                self.logger.error("Failed to write FAILED audit entry")
            raise

In [ ]:
# =======================================================
# APP
# =======================================================
class TotalAllPurposeSpendsApp:

    def __init__(self):
        catalog = dbutils.widgets.get("catalog")
        schema = dbutils.widgets.get("schema")
        ov_days = get_overlap_days(dbutils.widgets.get("overlap_days"), logger=logger)

        self.client = TotalAllPurposeSpendsClient(
            audit_table=build_table_fqn(catalog, schema, "dbspend360_audit_log"),
            cloud_cost_table=build_table_fqn(catalog, schema, "dbspend360_cloud_cost_explorer"),
            databricks_cost_table=build_table_fqn(catalog, schema, "dbspend360_all_purpose_dbu_cost"),
            target_table=build_table_fqn(catalog, schema, "dbspend360_total_all_purpose_spends"),
            error_log_table=build_table_fqn(catalog, schema, "dbspend360_error_log"),
            overlap_days=ov_days,
            logger=logger,
        )

    def run(self):
        self.client.build_total_all_purpose_spends()

In [ ]:
# =======================================================
# Execute
# =======================================================
app = TotalAllPurposeSpendsApp()
app.run()